# Siegle, Jia et al. (2021): the passive viewing survey, from the paper's DANDI deposits

**Paper.** Siegle JH, Jia X, Durand S, et al. (2021). Survey of spiking in the mouse visual system
reveals functional hierarchy. *Nature* 592, 86-92.
[doi:10.1038/s41586-020-03171-x](https://doi.org/10.1038/s41586-020-03171-x). The claims file was
written from the NIH author manuscript in PubMed Central (PMC10399640): main text, Methods, the
legends of Figures 1 to 4 and Extended Data Figures 1 to 10, and the availability statements.

**Target dandiset.** [DANDI:000253](https://dandiarchive.org/dandiset/000253/0.240923.1441),
version 0.240923.1441, the most recent published version (the paper cites no version of it; it
predates the dandiset). Its title and description are those of the OpenScope Global/Local Oddball
project; it names this paper as `IsSupplementTo`. The first section shows that it holds a
different experiment, so every claim the claims file marks `elsewhere` is tested on the archives
the paper names for its data.

**Where the paper's data are.** The Data availability statement deposits the 58 passive viewing
experiments on DANDI as dandisets 000021 and 000022. Pinned here:
[000021 version 0.251116.2246](https://dandiarchive.org/dandiset/000021/0.251116.2246) (Brain
Observatory 1.1 stimulus set) and
[000022 version 0.251116.2247](https://dandiarchive.org/dandiset/000022/0.251116.2247) (Functional
Connectivity stimulus set). The paper cites no version; these are the only published versions of
each (the other is the draft), so they are pinned. The per-unit tables and the behaviour metrics
tables of the paper's GitHub repository (AllenInstitute/neuropixels_platform_paper at commit
4f8bcc26) are read for the claims that the claims file says rest on them, and for the link claim.

This notebook tests the claims that do not need cross-correlograms. The CCG claims (Figure 2 and
Extended Data Figures 7, 8 and 10c to e) are in `ccg_siegle_2021.ipynb`.

| Claim | Data | Here |
|---|---|---|
| methods-sessions-passing-qc | 000021, 000022 | tested |
| methods-probes-ccf | 000021, 000022 | tested |
| ed1jk-units-per-experiment | 000021, 000022 | tested |
| fig1g-receptive-fields | 000021, 000022 | tested |
| link-unit-tables (added link) | 000021, 000022 and the paper's per-unit tables | tested |
| fig3c, ed9a, fig3f, ed9d, fig3i, ed9e, ed9fg, fig3l, ed9h, fig3-unit-counts, ed9bc, ed9ij | 000021, 000022 | tested |
| ed10b-layers | 000021, 000022 with the paper's layer table | tested |
| anatomical-hierarchy-order | the paper's code | tested (order only) |
| fig4e, fig4f, ed9k, ed9lmn, fig4h, ed9p | the paper's behaviour metrics tables | tested |
| methods-gaze-stability | 000021, 000022 | tested |
| fig4b-behaviour-performance | not deposited | blocked, data missing |
| CCG claims (nine) | 000021, 000022 | in `ccg_siegle_2021.ipynb` |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import io
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats
from statsmodels.stats.multitest import multipletests

import siegle as S

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()


def repo_csv(name):
    return pd.read_csv(io.StringIO(requests.get(f"{S.CODE_RAW}/data/{name}", timeout=60).text))

## The target dandiset, 000253

Its assets and metadata from the DANDI API, and the contents of one file, read without its data.

In [ ]:
tgt = S.list_assets(*S.TARGET)
meta253 = requests.get(f"{S.API}/dandisets/{S.TARGET[0]}/versions/{S.TARGET[1]}/").json()
print("name:", meta253["name"])
print("related:", [(r["relation"], r.get("name")) for r in meta253.get("relatedResource", [])])
print("assets:", len(tgt), "  subjects:", tgt.path.str.split("/").str[0].nunique(),
      "  sessions:", tgt.path.str.extract(r"ses-(\d+)")[0].nunique(),
      "  size (TB):", round(tgt.size_gb.sum() / 1000, 2))
print("files by kind:", tgt.path.str.extract(r"_(ogen|probe-\d)")[0].str.replace(r"probe-\d", "probe", regex=True).value_counts().to_dict())
f253 = S.open_asset(tgt[tgt.path.str.endswith("_ogen.nwb")].iloc[0].asset_id)
print("\nstimulus:", f253["general/stimulus"][()].decode())
print("interval tables:", sorted(f253["intervals"].keys()))
print("processing:", sorted(f253["processing"].keys()))
print("units:", f253["units/id"].shape[0], "columns:", sorted(k for k in f253["units"].keys() if not k.endswith("_index")))
f253.close()

The files of 000253 hold an oddball grating experiment (`init_grating`, receptive field mapping
with 1,944 presentations, spontaneous blocks) in 14 mice, with no full-field flashes, no 81-location
Gabor block of 3,645 presentations and no drifting gratings of 75 repeats, which the paper's
analyses use. The paper's recordings are not in it; the claims are tested on 000021 and 000022.

## Loading the paper's 58 sessions

For each session file of 000021 and 000022: the subject, the units table with its peak channel's
structure and CCF coordinates, the stimulus tables, the gaze in degrees, and, streamed for the
non-noise units of the eight visual areas and the hippocampus only, the spike times, from which the
helper computes each unit's receptive field test and area, times to first spike, drifting grating
rate and modulation index, and the two timescales. Four sessions are read at a time; one session
takes 30 to 60 s.

In [ ]:
assets = pd.concat([S.list_assets(d, v) for d, v in S.SOURCES.items()])
sessions_assets = assets[~assets.path.str.contains("probe")].reset_index(drop=True)
print("assets per dandiset:", assets.groupby("dandiset").size().to_dict(),
      " session files:", sessions_assets.groupby("dandiset").size().to_dict())
res = S.load_all(sessions_assets)
units = pd.concat([r[0] for r in res], ignore_index=True)
meta = pd.DataFrame([r[1] for r in res])
gaze = {r[1]["subject"]: r[2] for r in res}
print(f"loaded {len(meta)} sessions, {len(units)} units, in {(time.time() - started) / 60:.1f} min")

## Data check

In [ ]:
print("Sessions by dandiset and stimulus set:")
print(pd.crosstab(meta.dandiset, meta.session_type).to_string())
print("\nSubjects:", meta.subject.nunique(), " sessions:", len(meta))
print("\nGenotypes:")
print(meta.genotype.value_counts().to_string())
age_days = meta.age.str.extract(r"P(\d+)D")[0].astype(int)
print("\nSex:", meta.sex.value_counts().to_dict(), " age range (days):", age_days.min(), "to", age_days.max())
print("\nProbes per session:", meta.n_probes.value_counts().sort_index().to_dict(), " total:", meta.n_probes.sum())
print("\nUnits by quality:", units.quality.value_counts().to_dict())
print("Non-noise units by structure (eight areas and hippocampus):")
good = units[units.quality == "good"]
print(good.location.value_counts().reindex(S.AREAS + S.HPC).to_string())
print("\nUnits whose peak channel has no structure label:", int((units.location == "").sum()),
      " no CCF coordinate:", int(units.x.isna().sum()))

In [ ]:
print("Stimulus tables per session (counts, median durations in s):")
cols = ["n_flashes", "flash_duration", "flash_interval", "n_gabors", "gabor_duration", "gabor_positions", "dg_stimulus", "n_dg"]
print(meta.groupby("session_type")[cols].agg(lambda s: sorted(set(np.round(s, 3) if s.dtype.kind == "f" else s))).T.to_string())
print("\nFlash colours (dark -1, light +1) per session:", meta.flash_colors.astype(str).value_counts().to_dict())
print("\nSessions whose electrode y and z columns are equal on every channel:", int(meta.y_equals_z.sum()), "of", len(meta))
print("CCF ranges over units (um): x", np.nanmin(units.x), "to", np.nanmax(units.x),
      "; y", np.nanmin(units.y), "to", np.nanmax(units.y), "; z", np.nanmin(units.z), "to", np.nanmax(units.z))
print("\nSessions with gaze mapping:", int(meta.has_gaze.sum()))
m = units[units.p_value_rf.notna()]
print("\nUnits measured (non-noise, eight areas and hippocampus):", len(m))
desc = m[["p_value_rf", "area_rf", "tfs_all_250", "tfs_spont", "rate_pre", "firing_rate_dg", "mod_idx_dg",
          "timescale_it", "timescale_ac", "firing_rate", "snr"]].describe().T
print(desc[["count", "min", "50%", "max"]].round(4).to_string())
print("\nSpikes read:", int(meta.spikes_read.sum()), " seconds per session:", round(meta.seconds.min()), "to", round(meta.seconds.max()))

Checks of the description against the values. The 58 sessions split 32 and 26 between the Brain
Observatory 1.1 and Functional Connectivity sets, one session per mouse, as the Methods give. Every
session has 150 full-field flashes (75 dark, 75 light) of 250 ms every 2 s, and a Gabor block of 81
locations with 250 ms presentations, as the Methods describe; the Functional Connectivity sessions
have the drifting gratings of 4 orientations at 2 Hz with 75 repeats per contrast. Times are in
seconds and spike times share the clock of the stimulus tables: the receptive field test finds
significant fields in most visual area units and almost none in the hippocampus, and flash
latencies cluster in tens of ms (below), which a clock offset would not give. The electrodes table
gives the CCF position in x, y and z, but y and z are equal on every channel of every session, so
one of the three coordinates (the left-right one, z) is not the CCF value; x (anterior-posterior)
is used below only to say whether a probe was mapped.

## methods-sessions-passing-qc

> Of 87 mice entering the recording step, 61 passed session-level quality control and 58 resulted in
> NWB files; these 58 passive viewing experiments are the ones analysed.

Counts of sessions and of mice by genotype, from the session metadata (tolerance zero).

In [ ]:
geno = meta.genotype.str.split("/").str[0].str.replace("-IRES-Cre", "", regex=False)
counts = {"sessions": len(meta), "wt": int((geno == "wt").sum()), "Pvalb": int((geno == "Pvalb").sum()),
          "Sst": int((geno == "Sst").sum()), "Vip": int((geno == "Vip").sum())}
print(counts, " expected: sessions 58, wt 30, Pvalb 8, Sst 12, Vip 8")
print("Ai32 in every Cre line:", bool(meta[geno != "wt"].genotype.str.contains("Ai32").all()))

**Result: reproduced.** The two dandisets hold 58 sessions of 58 mice: 30 wild type, 8 Pvalb, 12
Sst and 8 Vip Cre lines crossed with Ai32, the counts of the Methods. The 87 and 61 are pipeline
counts before release and are not judged.

## methods-probes-ccf

> 284 of the 332 probes were mapped to the CCFv3.

As ExtDataFigure1jk.py counts it: per probe, the units passing ISI violations < 0.5, amplitude
cutoff < 0.1 and presence ratio > 0.95 (no quality filter in that loop); a probe is mapped when
the sum of their anterior-posterior CCF coordinates is positive (missing values skipped, as pandas
does). Reading (a), the 58 released sessions; reading (b) cannot be counted on released data.

In [ ]:
filt = (units.isi_violations < 0.5) & (units.amplitude_cutoff < 0.1) & (units.presence_ratio > 0.95)
probes = units.groupby(["session", "probe_id"]).size().rename("units").to_frame()
probes["units_filtered"] = units[filt].groupby(["session", "probe_id"]).size()
probes["ap_sum"] = units[filt].groupby(["session", "probe_id"]).x.sum()
probes = probes.fillna(0)
n_probes = int(meta.n_probes.sum())
print("probes (devices) in the 58 sessions:", n_probes, " probes with units:", len(probes))
print("probes mapped (sum of AP coordinates > 0):", int((probes.ap_sum > 0).sum()), " expected 284 of 332")
print("probes whose units all lack a CCF coordinate:", int((units.groupby(["session", "probe_id"]).x.count() == 0).sum()))
print("unmapped probes per session:", (probes.ap_sum <= 0).groupby(level=0).sum().value_counts().sort_index().to_dict())

**Result: reproduced.** The 58 sessions hold 332 probes, and 284 of them have a positive sum of
anterior-posterior CCF coordinates over their filtered units; the 48 others have no CCF coordinate
on any unit. Both counts equal the paper's under reading (a).

## ed1jk-units-per-experiment

> After quality filtering, 682 +/- 144 units were recorded per experiment, 119 +/- 48 per probe and
> 56 +/- 30 per visual area, with 6.1 +/- 1.1 visual areas sampled per experiment.

As ExtDataFigure1jk.py: the three filters and quality good per experiment and per area, the three
filters only per probe; areas with at least one unit; mean and numpy SD.

In [ ]:
f3 = (units.amplitude_cutoff < 0.1) & (units.presence_ratio > 0.95) & (units.isi_violations < 0.5)
sel1 = f3 & (units.quality == "good")
per_exp = units[sel1].groupby("session").size().reindex(meta.path.str.extract(r"ses-(\d+)")[0]).fillna(0)
per_probe = units[f3].groupby(["session", "probe_id"]).size()
per_probe = per_probe.reindex(units.groupby(["session", "probe_id"]).size().index).fillna(0)
pa = units[sel1 & units.location.isin(S.AREAS)].groupby(["session", "location"]).size()
areas_per_exp = pa.groupby(level=0).size().reindex(per_exp.index).fillna(0)
out = pd.DataFrame({"mean": [per_exp.mean(), per_probe.mean(), pa.mean(), areas_per_exp.mean()],
                    "sd": [per_exp.std(ddof=0), per_probe.std(ddof=0), pa.std(ddof=0), areas_per_exp.std(ddof=0)],
                    "n": [len(per_exp), len(per_probe), len(pa), len(areas_per_exp)],
                    "paper": ["682 +/- 144", "119 +/- 48", "56 +/- 30", "6.1 +/- 1.1"]},
                   index=["units per experiment", "units per probe", "units per area per experiment", "areas per experiment"])
print(out.round(2).to_string())
print("\nunits per experiment by stimulus set:")
st_of = meta.set_index(meta.path.str.extract(r"ses-(\d+)")[0]).session_type
print(per_exp.groupby(st_of).agg(["mean", "std", "min", "max"]).round(1).to_string())
print("probes counted here:", len(per_probe), " (probes with any unit)")

**Result: reproduced.** Units per experiment 686.2 +/- 144.6 (paper 682 +/- 144), per probe 128.8
+/- 51.1 (119 +/- 48), per area per experiment 56.2 +/- 30.3 (56 +/- 30), and 6.07 +/- 1.13 areas
per experiment (6.1 +/- 1.1); all five values are within their tolerances.

## fig1g-receptive-fields

> Units in the eight visual areas were highly visually responsive, 60% having a significant spatial
> receptive field within the monitor, against only 1.4% of units in the hippocampus (CA1, CA3 and
> dentate gyrus).

p < 0.01 by the chi-square shuffle test (1,000 shuffles, stopped early once p >= 0.01 is settled);
all non-noise units. The 60% pooled as Figure1g.py; the 1.4% under (a) pooled and (b) the mean
across mice of each mouse's fraction.

In [ ]:
g = units[(units.quality == "good")]
vis = g[g.location.isin(S.AREAS)]
hpc = g[g.location.isin(S.HPC)]
sig = lambda d: (d.p_value_rf < 0.01)
print("eight areas, pooled: %d of %d = %.4f" % (sig(vis).sum(), len(vis), sig(vis).mean()))
print("hippocampus (a) pooled: %d of %d = %.4f" % (sig(hpc).sum(), len(hpc), sig(hpc).mean()))
per_mouse = hpc.groupby("subject").apply(lambda d: sig(d).mean())
print("hippocampus (b) mean over %d mice of each mouse's fraction = %.4f (median %.4f, max %.4f)"
      % (len(per_mouse), per_mouse.mean(), per_mouse.median(), per_mouse.max()))
print("\nper area, pooled fraction and number of units:")
print(g[g.location.isin(S.AREAS + S.HPC)].groupby("location").apply(lambda d: pd.Series({"fraction": sig(d).mean(), "units": len(d)})).round(3).to_string())
print("\nunits whose p-value was not computed (no spike in any Gabor presentation):",
      int(vis.p_value_rf.isna().sum()), "visual,", int(hpc.p_value_rf.isna().sum()), "hippocampal")
print("threshold sensitivity, eight areas: p<0.001 %.3f, p<0.005 %.3f, p<0.02 %.3f" %
      tuple((vis.p_value_rf < t).mean() for t in (0.001, 0.005, 0.02)))
print("by stimulus set, eight areas:", vis.groupby("session_type").apply(lambda d: round(sig(d).mean(), 3)).to_dict())

**Result: partially reproduced.** 60.8% of the 43,990 non-noise units of the eight areas have
p < 0.01 (paper 60%, within tolerance). In the hippocampus 1.61% do pooled (reading a) and 1.74%
on the mean over 52 mice (reading b); both are outside 1.4% +/- 0.14%, though the contrast with
the visual areas is as large as the paper's.

## link-unit-tables

The paper's per-unit tables (`data/time_to_first_spike.csv`, `data/timescale_metrics.csv`,
`data/layer_info.csv`) are keyed by `ecephys_unit_id`. They are matched to the units of 000021 and
000022 by the units table `id`. The check: the share of the tables' ids found, and agreement of the
values recomputed here with the tables' values, with the procedures of the code that wrote them
(Figure3/get_flash_PSTH.py: first spike from 30 ms within 200 ms, 1 ms bins; timescale_calculation.py:
response decay timescale over 40 to 290 ms after onset), and the layer codes against the structures.

In [ ]:
tfs_csv = repo_csv("time_to_first_spike.csv")
ts_csv = repo_csv("timescale_metrics.csv")
layer_csv = repo_csv("layer_info.csv")
ids = set(units.unit_id)
good_ids = set(units[units.quality == "good"].unit_id)
for name, t in (("time_to_first_spike", tfs_csv), ("timescale_metrics", ts_csv), ("layer_info", layer_csv)):
    print(f"{name}: {len(t)} rows, {t.ecephys_unit_id.isin(ids).mean():.4f} of ids in the dandisets, "
          f"{t.ecephys_unit_id.isin(good_ids).mean():.4f} among non-noise units")
print("non-noise units of the dandisets absent from layer_info:", len(good_ids - set(layer_csv.ecephys_unit_id)))

a = units.merge(tfs_csv, left_on="unit_id", right_on="ecephys_unit_id")
a = a[a.tfs_all_200.notna() & a.time_to_first_spike_fl.notna()]
d = (a.tfs_all_200 - a.time_to_first_spike_fl).abs() * 1000
print(f"\ntime to first spike, 200 ms window, {len(a)} units with both: within 1 ms {np.mean(d <= 1):.4f}, "
      f"median |difference| {d.median():.2f} ms, 95th percentile {d.quantile(0.95):.2f} ms")
for c in ("tfs_all_250", "tfs_all_500"):
    dd = (a[c] - a.time_to_first_spike_fl).abs() * 1000
    print(f"  {c}: within 1 ms {np.mean(dd <= 1):.4f}, median |difference| {dd.median():.2f} ms")

b = units.merge(ts_csv, left_on="unit_id", right_on="ecephys_unit_id", suffixes=("", "_csv"))
b = b[b.timescale_ac_code.notna() & b.timescale_ac_csv.notna()]
rel = (b.timescale_ac_code - b.timescale_ac_csv).abs() / b.timescale_ac_csv.abs()
print(f"\nresponse decay timescale (40-290 ms), {len(b)} units with both: within 1% {np.mean(rel <= 0.01):.4f}")
print(f"units in timescale_metrics: {len(ts_csv)}; with a value here: {ts_csv.ecephys_unit_id.isin(units[units.timescale_ac_code.notna()].unit_id).sum()}")
bi = b[b.timescale_it_code.notna() & b.timescale_it_csv.notna()]
reli = (bi.timescale_it_code - bi.timescale_it_csv).abs() / bi.timescale_it_csv.abs()
print(f"intrinsic timescale (code window), {len(bi)} units: within 1% {np.mean(reli <= 0.01):.4f}")

c = units.merge(layer_csv, left_on="unit_id", right_on="ecephys_unit_id")
c["cortex"] = c.location.str.startswith("VIS")
print("\nlayer codes by structure class (rows: layer code; cortex = structure name starting with VIS):")
print(pd.crosstab(c.cortical_layer, c.cortex).to_string())
print("units with layer > 0 outside VIS* structures, by structure:",
      c[(c.cortical_layer > 0) & ~c.cortex].location.value_counts().head(8).to_dict())

**Result: partially reproduced.** Every id of the three tables is a non-noise unit of the
dandisets (664 non-noise units are absent from the layer table). The timescales agree: 99.3% of
23,352 units within 1% for the response decay timescale and 99.6% of 21,122 for the intrinsic
timescale. The time to first spike does not: with the code's 200 ms window 16.1% of 62,483 units
are within 1 ms, median difference 4.0 ms. The layer codes 1, 2 and 3 are also given to CA1, CA2
and CA3 units, so code 2 (L2/3) is not cortical only; with cortical_depth > 0, which the layer
analysis requires, only VIS* units remain (listed under ed10b).

## Figure 3: unit selection and the four metrics

The selection of Figure3.py: per area, p_value_rf < 0.01, area_rf < 2500, snr > 1,
firing_rate_dg > 0.1 Hz; then flash time to first spike < 100 ms for all four metrics; then, for the
response decay timescale, 1 < timescale < 300 ms, more than 50 spikes and SD of the estimate < 20.
Area means are over units (log10 for the modulation index), and Pearson's r is against the eight
hierarchy scores. Pairs of areas: two-sided rank-sum with 1e-5 added to each p, Benjamini-Hochberg
over the 28 pairs at 0.05, as Figure3.py.

In [ ]:
HS = np.array([S.HIERARCHY[a] for a in S.AREAS])
U3 = units[units.location.isin(S.AREAS) & (units.quality == "good")].copy()
base = (U3.p_value_rf < 0.01) & (U3.area_rf < 2500) & (U3.snr > 1) & (U3.firing_rate_dg > 0.1)


def area_corr(values, sel, f=lambda v: v, label=""):
    means = pd.Series({a: np.nanmean(f(values[sel & (U3.location == a)].values)) for a in S.AREAS})
    n = pd.Series({a: int((sel & (U3.location == a) & values.notna()).sum()) for a in S.AREAS})
    r, p = stats.pearsonr(HS, means.values)
    return means, n, r, p


def pairs_test(values, sel, f=lambda v: v):
    vals = {a: f(values[sel & (U3.location == a)].dropna().values) for a in S.AREAS}
    rows = []
    for i, a1 in enumerate(S.AREAS):
        for a2 in S.AREAS[i + 1:]:
            z, p = stats.ranksums(vals[a1], vals[a2])
            rows.append({"pair": f"{S.NAMES[a1]}-{S.NAMES[a2]}", "p": p + 1e-5})
    t = pd.DataFrame(rows)
    t["reject"], t["p_bh"] = multipletests(t.p.values, alpha=0.05, method="fdr_bh")[:2]
    return t


def judge_pairs(t, expected_ns):
    ns = set(t.pair[~t.reject])
    exp = {p if p in set(t.pair) else "-".join(p.split("-")[::-1]) for p in expected_ns}
    disagree = (ns - exp) | (exp - ns)
    return sorted(ns), sorted(disagree)


print("units per area: all non-noise, with RF, after the filters")
cnt = pd.DataFrame({"non_noise": U3.groupby("location").size(),
                    "p_rf<0.01": U3[U3.p_value_rf < 0.01].groupby("location").size(),
                    "after_filter": U3[base].groupby("location").size()}).reindex(S.AREAS)
cnt.index = [S.NAMES[a] for a in cnt.index]
print(cnt.to_string())
print("mice per area after the filter:", {S.NAMES[a]: U3[base & (U3.location == a)].subject.nunique() for a in S.AREAS})

### fig3c-latency-hierarchy

> The mean time to first spike after a full-field flash in each area is correlated with its
> anatomical hierarchy score.

Readings of the open detail (which flashes): (a) all, (b) dark, (c) light; window end 250 ms
(Methods), with 500 ms (functions/time_to_first_spike.py default) and 200 ms (get_flash_PSTH.py)
and the paper's table as variants. Each reading's own latency is used for the < 100 ms cut.

In [ ]:
rows = []
for col in ["tfs_all_250", "tfs_dark_250", "tfs_light_250", "tfs_all_500", "tfs_dark_500", "tfs_light_500", "tfs_all_200"]:
    sel = base & (U3[col] < 0.1)
    means, n, r, p = area_corr(U3[col] * 1000, sel)
    rows.append({"reading": col, "r": r, "p": p, "units": int(n.sum()), **{S.NAMES[a]: means[a] for a in S.AREAS}})
U3 = U3.merge(tfs_csv, left_on="unit_id", right_on="ecephys_unit_id", how="left").set_index(U3.index)
sel = base & (U3.time_to_first_spike_fl < 0.1)
means, n, r, p = area_corr(U3.time_to_first_spike_fl * 1000, sel)
rows.append({"reading": "paper table time_to_first_spike.csv", "r": r, "p": p, "units": int(n.sum()), **{S.NAMES[a]: means[a] for a in S.AREAS}})
lat = pd.DataFrame(rows).set_index("reading")
print("mean time to first spike per area (ms), Pearson r and p against hierarchy score (expected r 0.95 +/- 0.095):")
print(lat.round(4).to_string())
lat_sel = base & (U3.tfs_all_250 < 0.1)
print("\nper area range of unit latencies (ms), reading (a):",
      {S.NAMES[a]: (round(U3.tfs_all_250[lat_sel & (U3.location == a)].min() * 1000), round(U3.tfs_all_250[lat_sel & (U3.location == a)].max() * 1000)) for a in S.AREAS})
by_set = []
for stp in ("brain_observatory_1.1", "functional_connectivity"):
    m_, n_, r_, p_ = area_corr(U3.tfs_all_250 * 1000, lat_sel & (U3.session_type == stp))
    by_set.append({"set": stp, "r": r_, "p": p_, "units": int(n_.sum())})
print("reading (a) by stimulus set:", pd.DataFrame(by_set).round(4).to_dict("records"))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
ax.plot(HS, lat.loc["tfs_all_250", [S.NAMES[a] for a in S.AREAS]].values.astype(float), "o")
for a in S.AREAS:
    ax.annotate(S.NAMES[a], (S.HIERARCHY[a], lat.loc["tfs_all_250", S.NAMES[a]]), fontsize=8)
ax.set_xlabel("anatomical hierarchy score")
ax.set_ylabel("mean time to first spike (ms)")
plt.tight_layout()
print("plotted range (ms):", lat.loc["tfs_all_250", [S.NAMES[a] for a in S.AREAS]].astype(float).agg(["min", "max"]).round(2).to_dict())

**Result: reproduced.** With all flashes and the Methods' 250 ms window, the area means range
from 66.1 ms (V1) and 66.4 ms (LGN) to 75.1 ms (PM and AM), and r = 0.959, p = 0.0002 (paper 0.95). Dark flashes give
r = 0.904 and light flashes 0.933, both within tolerance with p < 0.01, so the readings agree. The
paper's own latency table gives r = 0.953.

### ed9a-latency-pairs

> Latencies differ significantly between all pairs of areas except LGN-V1, RL-LP, LP-AL and AM-PM.

In [ ]:
t = pairs_test(U3.tfs_all_250 * 1000, lat_sel)
ns, dis = judge_pairs(t, ["LGN-V1", "RL-LP", "LP-AL", "PM-AM"])
print(t.round(6).to_string(index=False))
print("\nnot significant:", ns)
print("pairs that disagree with the claim:", len(dis), dis)
for col in ("tfs_dark_250", "tfs_light_250", "tfs_all_500"):
    tt = pairs_test(U3[col] * 1000, base & (U3[col] < 0.1))
    print(f"variant {col}: not significant {judge_pairs(tt, [])[0]}")

**Result: partially reproduced.** Of the 28 pairs, 27 agree with the claim: LGN-V1, RL-LP, LP-AL
and PM-AM are not significant, but RL-AL is not significant either (corrected p = 0.52). With dark
flashes LGN-LM is the one pair that disagrees.

### fig3f-rf-size-hierarchy

> Receptive field size increases with anatomical hierarchy score.

In [ ]:
means, n, r, p = area_corr(U3.area_rf, lat_sel)
print("mean RF area (deg2) per area:", {S.NAMES[a]: round(means[a], 1) for a in S.AREAS})
print("units:", {S.NAMES[a]: int(n[a]) for a in S.AREAS})
print(f"Pearson r = {r:.4f}, p = {p:.3g} (expected 0.97 +/- 0.097)")
m2, n2, r2, p2 = area_corr(U3.area_rf, base)
print(f"variant without the latency cut: r = {r2:.4f}, p = {p2:.3g}")
for stp in ("brain_observatory_1.1", "functional_connectivity"):
    m_, n_, r_, p_ = area_corr(U3.area_rf, lat_sel & (U3.session_type == stp))
    print(f"{stp}: r = {r_:.4f}, p = {p_:.3g}")
print("range of unit RF areas (deg2):", U3.area_rf[lat_sel].min(), "to", U3.area_rf[lat_sel].max())

**Result: reproduced.** Mean RF area rises from 544 deg2 (LGN) to 905 deg2 (AM); r = 0.958,
p = 0.0002 (paper 0.97).

### ed9d-rf-size-pairs

> Receptive field sizes differ significantly between all pairs of areas except LM-RL.

In [ ]:
t = pairs_test(U3.area_rf, lat_sel)
ns, dis = judge_pairs(t, ["LM-RL"])
print(t.round(6).to_string(index=False))
print("\nnot significant:", ns)
print("pairs that disagree with the claim:", len(dis), dis)

**Result: partially reproduced.** LM-RL is not significant as stated, and two further pairs, LM-AL
(corrected p = 0.38) and RL-AL (0.067), are not significant either; 26 of 28 pairs agree.

### fig3i-modulation-index-hierarchy

> The modulation index of responses to drifting gratings is highest in LGN and decreases along the
> hierarchy.

In [ ]:
log10 = lambda v: np.log10(v)
means, n, r, p = area_corr(U3.mod_idx_dg, lat_sel, log10)
print("mean log10 MI per area:", {S.NAMES[a]: round(means[a], 3) for a in S.AREAS})
print("units:", {S.NAMES[a]: int(n[a]) for a in S.AREAS})
print(f"Pearson r = {r:.4f}, p = {p:.3g} (expected -0.89 +/- 0.089); highest area: {S.NAMES[means.idxmax()]}")
m2, n2, r2, p2 = area_corr(U3.mod_idx_dg, lat_sel)
print(f"variant, mean of MI: r = {r2:.4f}, p = {p2:.3g}; highest {S.NAMES[m2.idxmax()]}")
for stp in ("brain_observatory_1.1", "functional_connectivity"):
    m_, n_, r_, p_ = area_corr(U3.mod_idx_dg, lat_sel & (U3.session_type == stp), log10)
    print(f"{stp}: r = {r_:.4f}, p = {p_:.3g}, highest {S.NAMES[m_.idxmax()]}")
print("units with MI not computed or zero:", int((U3.mod_idx_dg[lat_sel].isna() | (U3.mod_idx_dg[lat_sel] <= 0)).sum()))
print("range of MI:", U3.mod_idx_dg[lat_sel].min(), "to", U3.mod_idx_dg[lat_sel].max())

**Result: reproduced.** Mean log10 MI is highest in LGN (0.628) and lowest in AM (-0.124);
r = -0.954, p = 0.0002 (paper -0.89).

### ed9e-modulation-index-pairs

> Modulation indices differ significantly between all pairs of areas except RL-AL and AM-PM.

In [ ]:
t = pairs_test(U3.mod_idx_dg.where(U3.mod_idx_dg > 0), lat_sel, log10)
ns, dis = judge_pairs(t, ["RL-AL", "PM-AM"])
print(t.round(6).to_string(index=False))
print("\nnot significant:", ns)
print("pairs that disagree with the claim:", len(dis), dis)

**Result: not reproduced.** PM-AM is not significant as stated, but RL-AL is significant
(corrected p = 0.0006), and LM-RL, LM-LP and RL-LP are not; 4 of 28 pairs disagree, more than the
two the criterion allows for a partial result.

### ed9fg-intrinsic-timescale-not-correlated

> The mean intrinsic timescale of spontaneous activity in each area is not correlated with the
> visual hierarchy.

Units of timescale_calculation.py (the Figure 3 filter, no latency cut); kept when the SD of the
estimate < 100 and at least 100 spikes. Variant from the code: 1.0 to 2.0 s after onset and
5 < timescale < 400 ms.

In [ ]:
it_sel = base & (U3.err_it < 100) & (U3.spike_count_it >= 100)
means, n, r, p = area_corr(U3.timescale_it, it_sel)
print("mean intrinsic timescale (ms):", {S.NAMES[a]: round(means[a], 1) for a in S.AREAS})
print("units:", {S.NAMES[a]: int(n[a]) for a in S.AREAS}, " fitted:", int(U3.timescale_it[base].notna().sum()), "of", int(base.sum()))
print(f"Pearson r = {r:.4f}, p = {p:.3g} (paper r = -0.24, p = 0.57)")
itc = base & (U3.err_it_code < 100) & (U3.spike_count_it_code >= 100) & (U3.timescale_it_code > 5) & (U3.timescale_it_code < 400)
m2, n2, r2, p2 = area_corr(U3.timescale_it_code, itc)
print(f"variant (code window and limits): r = {r2:.4f}, p = {p2:.3g}, units {int(n2.sum())}")
print("range of kept timescales (ms):", U3.timescale_it[it_sel].min(), "to", U3.timescale_it[it_sel].max())

**Result: reproduced.** The mean intrinsic timescale is not correlated with the hierarchy:
r = -0.158, p = 0.71 (paper -0.24, 0.57). The code's window gives r = -0.290, p = 0.49.

### fig3l-response-decay-timescale-hierarchy

> Higher-order areas have a longer response decay timescale during the flash response, and the area
> mean correlates with hierarchy score.

In [ ]:
def ac_sel(col, err, cnt):
    return lat_sel & (U3[col] < 300) & (U3[col] > 1) & (U3[cnt] > 50) & (U3[err] < 20)


ac_main = ac_sel("timescale_ac", "err_ac", "spike_count_ac")
means, n, r, p = area_corr(U3.timescale_ac, ac_main)
print("mean response decay timescale (ms), 0-250 ms:", {S.NAMES[a]: round(means[a], 1) for a in S.AREAS})
print("units:", {S.NAMES[a]: int(n[a]) for a in S.AREAS}, " total", int(n.sum()))
print(f"Pearson r = {r:.4f}, p = {p:.3g} (expected 0.86 +/- 0.086)")
ac_code = ac_sel("timescale_ac_code", "err_ac_code", "spike_count_ac_code")
m2, n2, r2, p2 = area_corr(U3.timescale_ac_code, ac_code)
print("variant 40-290 ms (code):", {S.NAMES[a]: round(m2[a], 1) for a in S.AREAS})
print(f"  r = {r2:.4f}, p = {p2:.3g}, units {int(n2.sum())}")
U3 = U3.merge(ts_csv[["ecephys_unit_id", "timescale_ac", "err_ac", "spike_count_ac"]].rename(columns=lambda c: c + "_csv" if c != "ecephys_unit_id" else "tsid"),
              left_on="unit_id", right_on="tsid", how="left").set_index(U3.index)
csv_sel = (base & (U3.time_to_first_spike_fl < 0.1) & (U3.timescale_ac_csv < 300) & (U3.timescale_ac_csv > 1)
           & (U3.spike_count_ac_csv > 50) & (U3.err_ac_csv < 20))
m3, n3, r3, p3 = area_corr(U3.timescale_ac_csv, csv_sel)
print(f"variant, the paper's tables for timescale and latency: r = {r3:.4f}, p = {p3:.3g}, units {int(n3.sum())}")

**Result: reproduced.** With the Methods' 0 to 250 ms window, r = 0.804, p = 0.016 (paper 0.86;
within tolerance), with PM and AM highest (53.3 and 54.6 ms). The code's 40 to 290 ms window gives
r = 0.876, and the paper's table 0.855.

### ed9h-response-decay-pairs

> Response decay timescales differ significantly between all pairs of areas except LM-AL and AM-PM.

In [ ]:
t = pairs_test(U3.timescale_ac, ac_main)
ns, dis = judge_pairs(t, ["LM-AL", "PM-AM"])
print(t.round(6).to_string(index=False))
print("\nnot significant:", ns)
print("pairs that disagree with the claim:", len(dis), dis)
tc = pairs_test(U3.timescale_ac_code, ac_code)
print("variant 40-290 ms, not significant:", judge_pairs(tc, [])[0])

**Result: not reproduced.** With the Methods' window PM-AM is not significant as stated, but LM-AL
is significant (corrected p = 0.018), and LGN-LM, LGN-AL and RL-LP are not; 4 of 28 pairs
disagree. With the code's 40 to 290 ms window only LGN-V1 disagrees.

### fig3-unit-counts

> The Figure 3 analyses use 15,713 units from 58 mice, and the response decay timescale 7,837 units.

In [ ]:
print("readings of the 15,713: (a) after the latency cut", int(lat_sel.sum()), "; (b) before it", int(base.sum()))
print("mice in (a):", U3[lat_sel].subject.nunique())
print("response decay timescale units (0-250 ms):", int(ac_main.sum()), "; 40-290 ms variant:", int(ac_code.sum()))
code_lat = base & (U3.tfs_all_200 < 0.1)
code_ac = (code_lat & (U3.timescale_ac_code < 300) & (U3.timescale_ac_code > 1) & (U3.spike_count_ac_code > 50)
           & (U3.err_ac_code < 20))
print("the code's procedures recomputed (first spike within 200 ms, get_flash_PSTH.py; decay timescale over 40-290 ms,",
      "timescale_calculation.py): after the latency cut", int(code_lat.sum()), "; decay timescale", int(code_ac.sum()))
print("with the paper's tables for latency and timescale:", int(csv_sel.sum()),
      "; after the latency cut from the paper's table:", int((base & (U3.time_to_first_spike_fl < 0.1)).sum()))

**Result: reproduced, resting on an assumption.** With the code's procedures recomputed (latency
within 200 ms as get_flash_PSTH.py, decay timescale over 40 to 290 ms as timescale_calculation.py),
the latency cut keeps 17,167 units (15,713 +/- 1,571) and the decay timescale 8,381 (7,837 +/- 784);
the paper's own tables give 15,853 and 7,798. With the Methods' windows the counts are 13,347 and
5,842, both outside tolerance, and before the latency cut 24,481 (reading b). The count is of
Figure3.py's selection, which reads the code's latency and timescale tables, so the code's
procedures are adopted.

### ed9bc-spontaneous-latency

> Time to first spike measured in the grey pre-stimulus interval is not correlated with hierarchy
> score but is strongly negatively correlated with baseline firing rate.

The same units as fig3c (reading (a)). Baseline rate readings: (a) the mean rate in the same 1 s
grey interval, (b) the unit table's overall rate.

In [ ]:
means, n, r, p = area_corr(U3.tfs_spont * 1000, lat_sel)
print("mean spontaneous time to first spike (ms):", {S.NAMES[a]: round(means[a], 1) for a in S.AREAS})
print(f"against hierarchy: r = {r:.4f}, p = {p:.3g} (paper r = 0.57, p = 0.14)")
rate_a = pd.Series({a: U3.rate_pre[lat_sel & (U3.location == a)].mean() for a in S.AREAS})
rate_b = pd.Series({a: U3.firing_rate[lat_sel & (U3.location == a)].mean() for a in S.AREAS})
for lab, rate in (("(a) pre-flash rate", rate_a), ("(b) overall rate", rate_b)):
    rr, pp = stats.pearsonr(rate.values, means.values)
    print(f"against baseline rate {lab}: r = {rr:.4f}, p = {pp:.3g} (expected -0.98 +/- 0.098); rates {rate.round(2).to_dict()}")
print("units without a spontaneous latency:", int(U3.tfs_spont[lat_sel].isna().sum()), "of", int(lat_sel.sum()))

**Result: reproduced.** The spontaneous time to first spike is not correlated with the hierarchy
(r = 0.290, p = 0.49; paper 0.57, 0.14) and is correlated with the pre-flash rate (r = -0.991,
p = 2e-06; paper -0.98). The overall rate gives r = -0.987.

### ed9ij-firing-rate-not-correlated

> Overall firing rate does not correlate with hierarchy score.

In [ ]:
means, n, r, p = area_corr(U3.firing_rate, lat_sel, log10)
print("mean log10 firing rate:", {S.NAMES[a]: round(means[a], 3) for a in S.AREAS})
print(f"Pearson r = {r:.4f}, p = {p:.3g}")
m2, n2, r2, p2 = area_corr(U3.firing_rate, lat_sel)
print(f"variant, mean rate: r = {r2:.4f}, p = {p2:.3g}; rates {m2.round(2).to_dict()}")

**Result: reproduced.** Mean log10 firing rate is not correlated with the hierarchy: r = -0.327,
p = 0.43; the untransformed mean gives r = -0.363, p = 0.38.

### ed10b-layers

> Compared with deep layers (5 and 6), superficial layers (2/3 and 4) have an earlier time to first
> spike, smaller receptive fields, a higher modulation index and a faster response decay timescale.

As the significance block of ExtDataFigure10.py, with the paper's layer table (linked above):
units with cortical_depth > 0 and the Figure 3 filters, cumulative cuts; superficial layers 2 and 4
(the codes the figure names L2/3 and L4), deep 5 and 6; difference of means; rank-sum.

In [ ]:
L = U3.merge(layer_csv, left_on="unit_id", right_on="ecephys_unit_id", how="left", suffixes=("", "_l")).set_index(U3.index)
Lall = units[(units.quality == "good")].merge(layer_csv, left_on="unit_id", right_on="ecephys_unit_id")
print("layer codes among non-noise units with depth > 0:", Lall[Lall.cortical_depth > 0].cortical_layer.value_counts().sort_index().to_dict())
# all cortical units with depth: the pool of the code is all structures, so the Figure 3 frame (eight areas) is widened
Ucx = units[(units.quality == "good") & units.p_value_rf.notna()].merge(layer_csv, left_on="unit_id", right_on="ecephys_unit_id")
print("units measured here with depth > 0:", int((Ucx.cortical_depth > 0).sum()),
      "; their structures:", Ucx[Ucx.cortical_depth > 0].location.value_counts().to_dict())


def layer_test(D, sup, deep, tfs="tfs_all_250", ts=("timescale_ac", "err_ac", "spike_count_ac")):
    sel = (D.p_value_rf < 0.01) & (D.cortical_depth > 0) & (D.area_rf < 2500) & (D.snr > 1) & (D.firing_rate_dg > 0.1)
    out = []
    specs = [("time to first spike (ms)", tfs, lambda v: v * 1000, "deep-sup"),
             ("RF area (deg2)", "area_rf", lambda v: v, "deep-sup"),
             ("log10 MI", "mod_idx_dg", np.log10, "sup-deep"),
             ("response decay timescale (ms)", ts[0], lambda v: v, "deep-sup")]
    for k, (name, col, f, sign) in enumerate(specs):
        if k == 0:
            sel = sel & (D[tfs] < 0.1)
        if k == 3:
            sel = sel & (D[ts[0]] < 300) & (D[ts[0]] > 1) & (D[ts[2]] > 50) & (D[ts[1]] < 20)
        a = f(D[col][sel & D.cortical_layer.isin(sup)].values)
        b = f(D[col][sel & D.cortical_layer.isin(deep)].values)
        a, b = a[np.isfinite(a)], b[np.isfinite(b)]
        diff = (b.mean() - a.mean()) if sign == "deep-sup" else (a.mean() - b.mean())
        out.append({"metric": name, "difference": diff, "as": sign, "p": stats.ranksums(a, b)[1],
                    "n_sup": len(a), "n_deep": len(b), "median_diff": (np.median(b) - np.median(a)) if sign == "deep-sup" else (np.median(a) - np.median(b))})
    return pd.DataFrame(out)


print("\nmain (layers 2/3 and 4 against 5 and 6, all cortical units with a depth):")
print(layer_test(Ucx, [2, 4], [5, 6]).to_string(index=False))
print("expected: 2.59 ms, 109 deg2, 0.09, 6.6 ms (each +/- 10%)")
print("\nvariant, the code's split (layer < 5 against >= 5):")
print(layer_test(Ucx, [1, 2, 3, 4], [5, 6]).to_string(index=False))
print("\nvariant, the six cortical areas only:")
print(layer_test(Ucx[Ucx.location.isin(S.AREAS[1:4] + S.AREAS[5:])], [2, 4], [5, 6]).to_string(index=False))
print("\nvariant, the code's procedures recomputed (first spike within 200 ms; decay timescale over 40-290 ms):")
print(layer_test(Ucx, [2, 4], [5, 6], "tfs_all_200", ("timescale_ac_code", "err_ac_code", "spike_count_ac_code")).to_string(index=False))
Ucsv = Ucx.merge(tfs_csv[["ecephys_unit_id", "time_to_first_spike_fl"]], on="ecephys_unit_id", how="left").merge(
    ts_csv[["ecephys_unit_id", "timescale_ac", "err_ac", "spike_count_ac"]].rename(
        columns={"timescale_ac": "ts_csv", "err_ac": "err_csv", "spike_count_ac": "cnt_csv"}), on="ecephys_unit_id", how="left")
print("\nvariant, the paper's tables for latency and timescale (as ExtDataFigure10.py reads them):")
print(layer_test(Ucsv, [2, 4], [5, 6], "time_to_first_spike_fl", ("ts_csv", "err_csv", "cnt_csv")).to_string(index=False))

**Result: partially reproduced, resting on an assumption.** With the code's procedures recomputed,
which ExtDataFigure10.py reads through the paper's tables, all four differences have the stated
sign with p < 1e-23: deep minus superficial 2.80 ms in latency, 113 deg2 in RF area and 6.39 ms in
decay timescale (each within tolerance), and superficial minus deep 0.127 in log10 MI (0.09
expected, outside tolerance): 3 of 4 parts. The paper's tables give the same 3 of 4 (2.55 ms,
110 deg2, 0.125, 6.69 ms). With the Methods' windows only the RF area is within tolerance (1.33 ms,
105 deg2, 0.117, 7.71 ms).

## anatomical-hierarchy-order

> In the anatomical hierarchy recomputed from ref. 3, LGN is lowest, then V1, with LM, RL, LP and AL
> at intermediate levels and PM and AM at the top.

The scores are read from Figure3/Figure3.py at the pinned commit.

In [ ]:
src = requests.get(f"{S.CODE_RAW}/Figure3/Figure3.py", timeout=60).text
block = re.search(r"hierarchy_score = \{(.*?)\}", src, re.S).group(1)
scores = {k: float(v) for k, v in re.findall(r"'(\w+)'\s*:\s*(-?[\d.]+)", block)}
order = sorted(scores, key=scores.get)
print("scores from the code:", scores)
print("order, low to high:", [S.NAMES[a] for a in order])
ok = (order[0] == "LGd" and order[1] == "VISp" and set(order[2:6]) == {"VISl", "VISrl", "LP", "VISal"}
      and set(order[6:]) == {"VISpm", "VISam"})
print("LGN lowest, V1 next, LM/RL/LP/AL intermediate, PM/AM top:", ok)
print("same scores as the helper uses:", scores == S.HIERARCHY)

**Result: reproduced.** The scores in the paper's code order the areas LGN, V1, LM, RL, LP, AL,
PM, AM, as stated. The bootstrap test of RL against LM needs the connectivity data of ref. 3 and was
not run.

## Figure 4 and Extended Data Figure 9k to p: the paper's behaviour metrics tables

`data/change_modulation_data.csv` (one row per unit, 4,057 units) and `data/decoding_data.csv`
(one row per session and area) at the pinned commit; per-region means and Pearson against the
hierarchy scores, as Figure4.py.

In [ ]:
cm = repo_csv("change_modulation_data.csv")
dec = repo_csv("decoding_data.csv")
REG = ["LGd", "V1", "LM", "RL", "LP", "AL", "PM", "AM"]
print("change modulation table:", cm.shape, " units per region:", cm.Region.value_counts().reindex(REG).to_dict())
print("decoding table:", dec.shape, " sessions per region:", dec.Region.value_counts().reindex(REG).to_dict())
print(cm.describe().T[["count", "min", "50%", "max"]].round(3).to_string())


def region_corr(df, col, f=np.nanmean):
    m = pd.Series({r: f(df[col][df.Region == r].values) for r in REG})
    r, p = stats.pearsonr(HS, m.values)
    return m, r, p

### fig4e-behaviour-latency-hierarchy

> During behaviour the first spike latency of image responses is correlated with the anatomical
> hierarchy score.

In [ ]:
m, r, p = region_corr(cm, "Time To First Spike")
print("mean time to first spike (ms):", m.round(2).to_dict())
print(f"Pearson r = {r:.4f}, p = {p:.3g}")

**Result: reproduced.** Mean latency ranges from 60.1 ms (LGd) to 77.2 ms (PM); r = 0.917,
p = 0.0013.

### fig4f-change-modulation-hierarchy

> During active behaviour the change modulation index is positive in every area and increases along
> the hierarchy from LGN to AM, and it is larger during active behaviour than during passive replay.

In [ ]:
ma, ra, pa_ = region_corr(cm, "Change Modulation Active")
mp, rp, pp = region_corr(cm, "Change Modulation Passive")
print("active means:", ma.round(4).to_dict())
print("passive means:", mp.round(4).to_dict())
print(f"active: Pearson r = {ra:.4f}, p = {pa_:.3g} (expected 0.83 +/- 0.083); passive r = {rp:.4f}, p = {pp:.3g}")
print("active mean > 0 in every area:", bool((ma > 0).all()))
print("(a) active > passive in every area:", bool((ma > mp).all()), " areas:", int((ma > mp).sum()), "of 8")
print("(b) active > passive in the mean over areas:", ma.mean() > mp.mean(), round(ma.mean(), 4), round(mp.mean(), 4))

**Result: reproduced.** Active CMI: r = 0.828, p = 0.011 (paper 0.83, 0.011); positive in all
eight areas (0.111 to 0.312); above the passive mean in all eight areas (mean 0.221 against 0.140).

### ed9k-cmi-hit-not-miss

> The change modulation index correlates with hierarchy on hit trials but not on miss trials.

In [ ]:
mh, rh, ph = region_corr(cm, "Change Modulation Hit")
mm, rm, pm = region_corr(cm, "Change Modulation Miss")
print(f"hit: r = {rh:.4f}, p = {ph:.3g} (expected 0.85 +/- 0.085); miss: r = {rm:.4f}, p = {pm:.3g} (paper 0.51, 0.2)")
print("hit means:", mh.round(4).to_dict())
print("miss means:", mm.round(4).to_dict())

**Result: reproduced.** Hit trials r = 0.853, p = 0.007; miss trials r = 0.510, p = 0.197, as the
paper reports.

### ed9lmn-other-responses-not-correlated

> Baseline firing rate, the response to the pre-change image and the response to the change image
> during the task are not correlated with hierarchy score.

In [ ]:
for col in ("Baseline Rate", "Pre-change Response", "Change Response"):
    for state in ("Active", "Passive"):
        m, r, p = region_corr(cm, f"{col} {state}")
        print(f"{col} {state}: r = {r:.4f}, p = {p:.3g}" + ("" if state == "Active" else "  (variant)"))

**Result: reproduced.** None of the three active measures is correlated with the hierarchy
(baseline r = -0.584, p = 0.13; pre-change r = -0.239, p = 0.57; change r = 0.107, p = 0.80). Nor are
the passive ones (p = 0.079 to 0.66).

### fig4h-decoder-behaviour-hierarchy

> The trial-wise covariation of change decoder output with the mouse's choices increases along the
> hierarchy, starting with no correlation in LGN.

In [ ]:
col = "Correleation of decoder prediction and mouse behavior"
m = dec.groupby("Region")[col].mean().reindex(REG)
sem = dec.groupby("Region")[col].agg(lambda v: np.std(v) / np.sqrt(len(v))).reindex(REG)
r, p = stats.pearsonr(HS, m.values)
print("mean per region:", m.round(4).to_dict())
print(f"Pearson r = {r:.4f}, p = {p:.3g} (expected 0.88 +/- 0.088)")
print(f"LGN mean +/- s.e.m.: {m['LGd']:.4f} +/- {sem['LGd']:.4f}; interval includes zero: {m['LGd'] - sem['LGd'] <= 0 <= m['LGd'] + sem['LGd']}")

**Result: reproduced.** r = 0.881, p = 0.0038 (paper 0.88, 0.004); LGN -0.015 +/- 0.032 includes
zero.

### ed9p-decoders-above-chance

> Change decoders trained on 20 units performed better than chance in all eight areas, and decoder
> accuracy was not correlated with hierarchy score.

In [ ]:
acc = dec.groupby("Region")["Decoder accuracy"]
t = pd.DataFrame({"mean": acc.mean(), "sem": acc.agg(lambda v: np.std(v) / np.sqrt(len(v))), "n": acc.size(),
                  "t_p_one_sided": acc.agg(lambda v: stats.ttest_1samp(v, 0.5, alternative="greater").pvalue)}).reindex(REG)
t["(a) mean-sem > 0.5"] = t["mean"] - t["sem"] > 0.5
t["(b) p < 0.05"] = t.t_p_one_sided < 0.05
print(t.round(5).to_string())
r, p = stats.pearsonr(HS, t["mean"].values)
print(f"accuracy against hierarchy: r = {r:.4f}, p = {p:.3g}")

**Result: reproduced.** Every area is above 0.5 under both readings (means 0.688 to 0.835; the
smallest, LGN with 3 sessions, has one-sided p = 0.0006), and accuracy is not correlated with the
hierarchy (r = 0.566, p = 0.14).

## methods-gaze-stability

> Across 50 mice with eye tracking, 95% of gaze locations fell within 6.4 +/- 2.1 degrees of the
> mean, with a maximum of 13.6 degrees.

Filtered gaze in degrees (`screen_coordinates_spherical`), frames with a finite value; per mouse the
95th percentile of the distance from the mean gaze: (a) radial, (b) the larger of the two axes'
95th percentiles of the absolute deviation. Raw gaze is a variant.

In [ ]:
rows = []
for subj, g in gaze.items():
    for kind, xy in g.items():
        rad, ax_, n = S.gaze_radius(xy)
        rows.append({"subject": subj, "kind": kind, "radial": rad, "per_axis": ax_, "frames": n, "of": len(xy)})
gz = pd.DataFrame(rows)
for kind in ("filtered", "raw"):
    d = gz[gz.kind == kind]
    print(f"{kind}: mice {len(d)}; (a) radial {d.radial.mean():.2f} +/- {d.radial.std(ddof=0):.2f} (sd ddof 0), "
          f"sd ddof 1 {d.radial.std():.2f}, max {d.radial.max():.2f}; (b) per axis {d.per_axis.mean():.2f} +/- {d.per_axis.std(ddof=0):.2f}, max {d.per_axis.max():.2f}")
d = gz[gz.kind == "filtered"]
print("frames kept per mouse (fraction):", round((d.frames / d["of"]).min(), 3), "to", round((d.frames / d["of"]).max(), 3))
print("radial values, sorted:", np.round(np.sort(d.radial.values), 2).tolist())

**Result: reproduced.** Over the 50 mice with gaze mapping, the radial 95% distance is
6.59 +/- 2.18 degrees with a maximum of 14.29 (paper 6.4 +/- 2.1 and 13.6, both within tolerance);
the per-axis reading gives 6.21 +/- 2.13 and 13.37.

## Summary

Claims tested in this notebook; the CCG claims are summarized in `ccg_siegle_2021.ipynb`.

| Claim | Data | Outcome |
|---|---|---|
| methods-sessions-passing-qc | 000021, 000022 | reproduced |
| methods-probes-ccf | 000021, 000022 | reproduced (284 of 332) |
| ed1jk-units-per-experiment | 000021, 000022 | reproduced |
| fig1g-receptive-fields | 000021, 000022 | partially reproduced (60.8%; hippocampus 1.6%) |
| link-unit-tables | 000021, 000022 and the paper's tables | partially reproduced (ids and timescales match; latencies do not) |
| fig3c-latency-hierarchy | 000021, 000022 | reproduced, r = 0.959 |
| ed9a-latency-pairs | 000021, 000022 | partially reproduced (27 of 28 pairs) |
| fig3f-rf-size-hierarchy | 000021, 000022 | reproduced, r = 0.958 |
| ed9d-rf-size-pairs | 000021, 000022 | partially reproduced (26 of 28) |
| fig3i-modulation-index-hierarchy | 000021, 000022 | reproduced, r = -0.954 |
| ed9e-modulation-index-pairs | 000021, 000022 | not reproduced (24 of 28) |
| ed9fg-intrinsic-timescale-not-correlated | 000021, 000022 | reproduced (p = 0.71) |
| fig3l-response-decay-timescale-hierarchy | 000021, 000022 | reproduced, r = 0.804 |
| ed9h-response-decay-pairs | 000021, 000022 | not reproduced (24 of 28) |
| fig3-unit-counts | 000021, 000022 | reproduced with the code's procedures (17,167 and 8,381); not with the Methods' windows |
| ed9bc-spontaneous-latency | 000021, 000022 | reproduced |
| ed9ij-firing-rate-not-correlated | 000021, 000022 | reproduced |
| ed10b-layers | 000021, 000022 with the layer table | partially reproduced (3 of 4 parts with the code's procedures; MI difference 0.127) |
| anatomical-hierarchy-order | the paper's code | reproduced (order only) |
| fig4e, fig4f, ed9k, ed9lmn, fig4h, ed9p | the paper's behaviour tables | reproduced |
| methods-gaze-stability | 000021, 000022 | reproduced |
| fig4b-behaviour-performance | not deposited | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper took the receptive field, modulation index
and drifting grating rate of each unit from the AllenSDK's precomputed metrics, which are not in
the NWB files; they are recomputed here from the spike times as the Methods describe them. The
preferred grating condition (the stimulus condition with the highest mean count) and
firing_rate_dg (spikes over all grating presentations divided by their duration) are assumptions.
Latencies and response decay timescales follow the Methods' windows (250 ms; 0 to 250 ms); the
code's windows (200 ms in the script that wrote the paper's latency table; 40 to 290 ms) are shown
as variants, and they change the unit counts and some pairwise tests. The paper's latency table
does not match latencies recomputed with its own script's procedure (median difference 4 ms).

What was found about the data. Dandiset 000253 is an OpenScope oddball experiment and holds none of
the paper's recordings, though its metadata names the paper as `IsSupplementTo`. The paper's data
are in 000021 and 000022, which match the paper's counts of sessions, genotypes and probes. In
every session the electrodes' `y` and `z` columns are identical, so the left-right CCF coordinate
is lost. The units tables carry no stimulus metrics (p_value_rf, area_rf, mod_idx_dg), which the
paper's figures use. The behaviour experiments of Figure 4 are not deposited as recordings; only
the paper's per-unit and per-session metric tables are public.